# Criando conexao com phoenix


````python
## para acessar localmente

session = px.launch_app()

tracer_provider = register(
  project_name="Code-Deep-Agent",
  endpoint="http://localhost:6006/v1/traces",
  auto_instrument=True
)
```


In [1]:
from phoenix.otel import register
import phoenix as px
import nest_asyncio


import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#session = px.launch_app()

In [3]:
"""tracer_provider = register(
  project_name="Code-Deep-Agent",
  endpoint="http://localhost:6006/v1/traces",
  auto_instrument=True
)"""

'tracer_provider = register(\n  project_name="Code-Deep-Agent",\n  endpoint="http://localhost:6006/v1/traces",\n  auto_instrument=True\n)'

In [4]:
import os
tracer_provider = register(
  project_name="Code-Deep-Agent",
  endpoint="https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces",
  auto_instrument=True,
  api_key=os.getenv("PHOENIX_API_KEY")
  
)

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\phoenix\otel\otel.py:434: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


OpenTelemetry Tracing Details
|  Phoenix Project: Code-Deep-Agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
from openinference.instrumentation.langchain import LangChainInstrumentor
import os


os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/sehnemjeferson"
#os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006/v1/traces"
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.getenv('PHOENIX_API_KEY')}"

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

In [6]:
#px_client = px.Client()

In [7]:
#px.launch_app().view()

# Coletando os dados

```python

## Coletando os dados completos

from datasets import load_dataset
from tqdm import tqdm
import pandas as pd

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)
    
inputs = [item['inputs']['aswer_code'] for item in data_set_code_langsmith]

outputs = [item['outputs']['response_code'] for item in data_set_code_langsmith]

dataset_df = pd.DataFrame(data={"query": inputs, "responses": outputs})

from phoenix.client import AsyncClient, Client
import pandas as pd

px_client = AsyncClient(base_url="https://app.phoenix.arize.com/s/sehnemjeferson", api_key=os.getenv("PHOENIX_API_KEY"))

dataset = await px_client.datasets.create_dataset(
    dataframe=dataset_df,
    name="dataset_code",
    input_keys=["query"],
    output_keys=["responses"],
)

```




In [8]:
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)
    
inputs = [item['inputs']['aswer_code'] for item in data_set_code_langsmith]

outputs = [item['outputs']['response_code'] for item in data_set_code_langsmith]

dataset_df = pd.DataFrame(data={"query": inputs, "responses": outputs})

Problems: 100%|██████████| 164/164 [00:00<00:00, 5436.47problem/s]


 Selecionando uma amostra aleatoria

In [9]:
tamanho_amostra = 5

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(dataset_df)), tamanho_amostra)
dataset_df_aleatorios = dataset_df.iloc[index_aleatorios]

```Inviando os dados para o PHOENIX CLOUD.```

In [10]:
from phoenix.client import AsyncClient, Client
import pandas as pd
import os

px_client = AsyncClient(base_url="https://app.phoenix.arize.com/s/sehnemjeferson", api_key=os.getenv("PHOENIX_API_KEY"))


In [11]:
try:
    dataset = await px_client.datasets.create_dataset(
        dataframe=dataset_df_aleatorios,
        name=f"dataset_code_aleatorios_{tamanho_amostra}",
        input_keys=["query"],
        output_keys=["responses"],
    )
except Exception as e:
    print(e)


INFO:phoenix.client.resources.datasets:Uploading dataset...
INFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/upload?sync=true "HTTP/1.1 409 Conflict"


Dataset upload failed: Dataset with the same name already exists: name='dataset_code_aleatorios_5'


 Coletando os dados do experimento

```python
 dataset = await px_client.datasets.get_dataset(dataset="dataset_code_aleatorios_5", version_id="RGF0YXNldFZlcnNpb246NQ==")

```

# Fazendo experimentos com o agente

## Testando varios modelos 5 dados aleatorios



In [12]:
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact
from langchain_core.messages import HumanMessage
from phoenix.client import AsyncClient, Client
import os
px_client = AsyncClient(base_url="https://app.phoenix.arize.com/s/sehnemjeferson", api_key=os.getenv("PHOENIX_API_KEY"))

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found qwen/qwen3-coder-480b-a35b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [13]:
code_agent = CodeAgentReact(model="qwen/qwen3-next-80b-a3b-instruct", model_provider="nvidia")

agent = code_agent.create_agent()

async def agent_avaliado(input):
    
    question = input["query"]
    
    
    answer = await agent.ainvoke(
    {
        "messages": 
            [HumanMessage(role="user",
                          content=question)],
        "todos": [],
    }
)
    return answer['messages'][-1].content

INFO:code_agent.creat_react_code_agent.code_agent_react:CodeAgentReact inicializado: provider=nvidia, checkpointer=False
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found qwen/qwen3-next-80b-a3b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\chat_models.py:715: UserWarning: Model 'qwen/qwen3-next-80b-a3b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso


## Avaliando a conciseness

In [14]:
def conciseness(output: dict) -> bool:
    if isinstance(output, dict):
        words = outputs["responses"].split(" ")
    if isinstance(output, str):
        words = output.split(" ")
    else:
        words = []
    return 1.0 if len(words) <= 500 else 0.0

## Contagem de ferramenta

In [15]:
from langchain_core.messages import HumanMessage, ToolMessage
def cont_tool(response):
    tools = code_agent.create_tools()
    tools_names = [tool.name for tool in tools]
    
    contadores = {name: 0 for name in tools_names}
    
    for message in response['messages']:
        if isinstance(message, ToolMessage):
            if message.name in contadores:
                contadores[message.name] += 1
    
    return contadores

def tool_write_code(response):
    tools = cont_tool(response)
    tools_score = tools["write_code"]/1
    return tools_score



## Correctness score

In [16]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score for the correctness of the answer, of an answer between 0 and 1")

In [17]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider
from typing import Any, Dict
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.cerebras import CerebrasProvider
import os

#@create_evaluator(kind="llm")
async def correctness(input: dict, output, expected: Dict[str, Any]) -> bool:
    
    """if isinstance(output, dict):
        output = output["messages"][-1].content"""
    
    
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {input}
    </input>

    <output>
        {output}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {reference_outputs}
    </reference_outputs>
    """.format(input=input["query"], output=output, reference_outputs = expected["responses"])

    model = GoogleModel('models/gemini-2.0-flash-lite')
    
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = await asyncio.to_thread(agent.run_sync, prompt)
    except Exception as e:
        try:
            logger.error(f"Erro do tipo: {e}")
            model = GoogleModel('models/gemini-2.0-flash')
            agent = Agent(model, output_type=CorrectnessScore)
            response = await asyncio.to_thread(agent.run_sync, prompt)
        except Exception as e:
            logger.error(f"Erro do tipo: {e}")
            model = model = OpenAIChatModel(
                'qwen-3-235b-a22b-instruct-2507',
                provider=CerebrasProvider(api_key=os.getenv("CEREBRAS_API_KEY")),
            )
            agent = Agent(model, output_type=CorrectnessScore)
            response = await asyncio.to_thread(agent.run_sync, prompt)
    
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response

## Code correctness

In [18]:
CODE_CORRECTNESS_PROMPT_WITH_REFERENCE_OUTPUTS = """You are an expert code reviewer evaluating code for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct code solution:
  - Solves the problem completely as specified in the input
  - Should contain only valid code without any additional text
  - Handles all edge cases appropriately
  - Contains absolutely no bugs or logical errors
  - Uses efficient and appropriate algorithms/data structures
  - Follows language-specific best practices
  - Has correct syntax and would compile/run without errors

  When scoring, you should penalize:
  - Logical errors or bugs that would cause incorrect behavior
  - Missing edge case handling
  - Overly inefficient implementations when better approaches exist
  - Incomplete solutions that don't address all requirements
  - Syntax errors that would prevent compilation/execution
  - Security vulnerabilities or unsafe practices
  - Additional text that is not code
</Rubric>

<Instructions>
  - Carefully analyze both the output code and the initial input query
  - Meticulously check for functional correctness and completeness
  - Focus on whether the code would work correctly rather than style preferences
  - Compare the output with the reference output to verify correctness
  - The reference output represents the expected behavior or result
  - Code that produces results matching the reference output should be scored higher
  - Consider edge cases where the code might produce correct results for the given examples but fail in other scenarios
</Instructions>

<Reminder>
  The goal is to evaluate whether the code correctly solves the given problem and produces output that matches the reference.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_output>
{reference_outputs}
</reference_output>
"""

In [19]:
from pydantic import BaseModel, Field
from code_agent.get_routem_llm.routem_llm import LlmRouter
from typing import Dict, Any

class CodeOutput(BaseModel):
    """Schema for code solutions to questions about LCEL."""
    code: str = Field(description="You should stract the code solution from the response.")

async def extract_code_from_response(response: str) -> str:
    
    """Extract code solution from the model response."""
    
    router_structured = LlmRouter(response, CodeOutput) 

    response_code_formatted = await router_structured.llm_router()
    
    if isinstance(response_code_formatted, dict):
        response_code_formatted = response_code_formatted['code']
        
    elif isinstance(response_code_formatted, CodeOutput):
        response_code_formatted = response_code_formatted.code
    else:
        response_code_formatted = ""
    
    return response_code_formatted

async def correctness_code(input: dict, output, expected: Dict[str, Any]) -> bool:
    
    """if isinstance(output, dict):
        output = output["messages"][-1].content"""
    
    output_formatted = await extract_code_from_response(output)
    
    prompt = CODE_CORRECTNESS_PROMPT_WITH_REFERENCE_OUTPUTS.format(inputs=input["query"], outputs=output_formatted, reference_outputs=expected["responses"])

    model = GoogleModel('models/gemini-2.0-flash-lite')
    
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = await asyncio.to_thread(agent.run_sync, prompt)
    except Exception as e:
        try:
            logger.error(f"Erro do tipo: {e}")
            model = GoogleModel('models/gemini-2.0-flash')
            agent = Agent(model, output_type=CorrectnessScore)
            response = await asyncio.to_thread(agent.run_sync, prompt)
        except Exception as e:
            logger.error(f"Erro do tipo: {e}")
            model = model = OpenAIChatModel(
                'qwen-3-235b-a22b-instruct-2507',
                provider=CerebrasProvider(api_key=os.getenv("CEREBRAS_API_KEY")),
            )
            agent = Agent(model, output_type=CorrectnessScore)
            response = await asyncio.to_thread(agent.run_sync, prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
        
    return response

## Coletando dados

In [20]:
# Get the current dataset version. You can omit the version for the latest.
try:
    dataset = await px_client.datasets.get_dataset(dataset="dataset_code_aleatorios_5", version_id="RGF0YXNldFZlcnNpb246MTI=")
except Exception as e:
    print(e)

INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets?name=dataset_code_aleatorios_5 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMg%3D%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMg%3D%3D/examples?version_id=RGF0YXNldFZlcnNpb246MTI%3D "HTTP/1.1 200 OK"


## Avaliando

In [21]:
"""from phoenix.client.experiments import run_experiment, async_run_experiment
experiment = await async_run_experiment(
        dataset=dataset,
        task=agent_avaliado,
        experiment_name="write-code-mudado",
        evaluators=[correctness_code, conciseness, correctness],
        timeout=500,
        
    )"""

'from phoenix.client.experiments import run_experiment, async_run_experiment\nexperiment = await async_run_experiment(\n        dataset=dataset,\n        task=agent_avaliado,\n        experiment_name="write-code-mudado",\n        evaluators=[correctness_code, conciseness, correctness],\n        timeout=500,\n\n    )'

In [22]:
from phoenix.client.experiments import run_experiment, async_run_experiment
rodar = False
if rodar:
    modelos = [
        #{"model": "openai/gpt-oss-20b",  "provider": "groq"},
        #{"model": "meta-llama/llama-4-scout-17b-16e-instruct",  "provider": "groq"},
        #{"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "groq"},
        #{"model": "qwen/qwen3-next-80b-a3b-instruct",  "provider": "nvidia"},
        #{"model": "deepseek-ai/deepseek-v3.1",  "provider": "nvidia"},
        #{"model": "microsoft/phi-4-mini-instruct",  "provider": "nvidia"},
        #{"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "nvidia"},
        #{"model": "meta/llama-4-scout-17b-16e-instruct",  "provider": "nvidia"},
        #{"model": "openai/gpt-oss-120b",  "provider": "nvidia"},
        #{"model": "moonshotai/kimi-k2-instruct",  "provider": "nvidia"},
        #{"model": "meta/llama-3.3-70b-instruct",  "provider": "nvidia"},
        #{"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",  "provider": "nvidia"},
        #{"model": "nvidia/llama-3.1-nemotron-ultra-253b-v1",  "provider": "nvidia"},
        #{"model": "meta/llama-3.1-405b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.1-nemotron-nano-4b-v1.1",  "provider": "nvidia"},
        {"model": "ibm/granite-3.3-8b-instruct",  "provider": "nvidia"},
        {"model": "nv-mistralai/mistral-nemo-12b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.3-nemotron-super-49b-v1",  "provider": "nvidia"},
        {"model": "mistralai/mistral-small-3.1-24b-instruct-2503",  "provider": "nvidia"},
        {"model": "qwen/qwq-32b",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-8b-instruct",  "provider": "nvidia"},
        {"model": "mistralai/mistral-nemotron",  "provider": "nvidia"},
        {"model": "meta/llama-3.2-3b-instruct",  "provider": "nvidia"},
        {"model": "openai/gpt-oss-20b",  "provider": "nvidia"},
        {"model": "mistralai/mistral-large-2-instruct",  "provider": "nvidia"},
        {"model": "deepseek-ai/deepseek-r1-0528",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-70b-instruct",  "provider": "nvidia"}
        
        
    ]
    
    for modelo in modelos:
        
        print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provider']}")
        
        code_agent = CodeAgentReact(model=modelo["model"], model_provider=modelo["provider"])

        agent = code_agent.create_agent()


        async def agent_avaliado(input):
            
            question = input["query"]
            
            
            answer = await agent.ainvoke(
            {
                "messages": 
                    [HumanMessage(role="user",
                                content=question)],
                "todos": [],
            }
        )
            return answer['messages'][-1].content
        
        experiment = await async_run_experiment(
        dataset=dataset,
        task=agent_avaliado,
        experiment_name=f"{dataset.name}-{modelo['model']}-{modelo['provider']}-correcao-state",
        evaluators=[correctness_code, conciseness, correctness],
        timeout=500,
        
    )
    

## Testando os melhores modelos com 15 dados aleatorios

In [23]:
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)
    
inputs = [item['inputs']['aswer_code'] for item in data_set_code_langsmith]

outputs = [item['outputs']['response_code'] for item in data_set_code_langsmith]

dataset_df = pd.DataFrame(data={"query": inputs, "responses": outputs})

Problems: 100%|██████████| 164/164 [00:00<00:00, 2743.33problem/s]


In [24]:
tamanho_amostra = 20

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(dataset_df)), tamanho_amostra)
dataset_df_aleatorios = dataset_df.iloc[index_aleatorios]

In [25]:
from phoenix.client import AsyncClient, Client
import pandas as pd
import os

px_client = AsyncClient(base_url="https://app.phoenix.arize.com/s/sehnemjeferson", api_key=os.getenv("PHOENIX_API_KEY"))

In [26]:
try:
    dataset = await px_client.datasets.create_dataset(
        dataframe=dataset_df_aleatorios,
        name=f"dataset_code_aleatorios_{tamanho_amostra}",
        input_keys=["query"],
        output_keys=["responses"],
    )
except Exception as e:
    print(e)

INFO:phoenix.client.resources.datasets:Uploading dataset...
INFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/upload?sync=true "HTTP/1.1 409 Conflict"


Dataset upload failed: Dataset with the same name already exists: name='dataset_code_aleatorios_20'


In [27]:

try:
    dataset = await px_client.datasets.get_dataset(dataset="dataset_code_aleatorios_20", version_id="RGF0YXNldFZlcnNpb246MTE=")
except Exception as e:
    print(e)

INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets?name=dataset_code_aleatorios_20 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMQ%3D%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMQ%3D%3D/examples?version_id=RGF0YXNldFZlcnNpb246MTE%3D "HTTP/1.1 200 OK"


In [ ]:
from phoenix.client.experiments import run_experiment, async_run_experiment
rodar = True
if rodar:
    modelos = [
        #{"model": "qwen/qwen3-next-80b-a3b-instruct",  "provider": "nvidia"},
        #{"model": "meta-llama/llama-4-scout-17b-16e-instruct",  "provider": "groq"},
        #{"model": "deepseek-ai/deepseek-v3.1",  "provider": "nvidia"},
        #{"model": "microsoft/phi-4-mini-instruct",  "provider": "nvidia"},
        #{"model": "meta/llama-4-scout-17b-16e-instruct",  "provider": "nvidia"},
        #{"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",  "provider": "nvidia"},
        #{"model": "nvidia/llama-3.1-nemotron-ultra-253b-v1",  "provider": "nvidia"},
        #{"model": "meta/llama-3.1-405b-instruct",  "provider": "nvidia"},
        {"model": "ibm/granite-3.3-8b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.3-nemotron-super-49b-v1",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-8b-instruct",  "provider": "nvidia"},
        {"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "nvidia"},
        
        
    ]
    
    for modelo in modelos:
        
        print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provider']}")
        
        code_agent = CodeAgentReact(model=modelo["model"], model_provider=modelo["provider"])

        agent = code_agent.create_agent()


        async def agent_avaliado(input):
            
            question = input["query"]
            
            
            answer = await agent.ainvoke(
            {
                "messages": 
                    [HumanMessage(role="user",
                                content=question)],
                "todos": [],
            }
        )
            return answer['messages'][-1].content
        
        experiment = await async_run_experiment(
        dataset=dataset,
        task=agent_avaliado,
        experiment_name=f"{dataset.name}-{modelo['model']}-{modelo['provider']}-correcao-state",
        evaluators=[correctness_code, conciseness, correctness],
        timeout=500,
        
    )
    

INFO:code_agent.creat_react_code_agent.code_agent_react:CodeAgentReact inicializado: provider=nvidia, checkpointer=False
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso
INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso


Rodando o agente com o modelo ibm/granite-3.3-8b-instruct do provedor nvidia


INFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMQ==/experiments "HTTP/1.1 200 OK"


🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/s/sehnemjeferson/datasets/RGF0YXNldDoxMQ==/experiments
🔗 View this experiment: https://app.phoenix.arize.com/s/sehnemjeferson/datasets/RGF0YXNldDoxMQ==/compare?experimentId=RXhwZXJpbWVudDozNjk=


running tasks |          | 0/20 (0.0%) | ⏳ 00:00<? | ?it/sINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNjk=/runs "HTTP/1.1 200 OK"
running tasks |▌         | 1/20 (5.0%) | ⏳ 00:25<08:02 | 25.38s/itINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNjk=/runs "HTTP/1.1 200 OK"
running tasks |█         | 2/20 (10.0%) | ⏳ 00:27<03:33 | 11.84s/itINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNjk=/runs "HTTP/1.1 200 OK"
running tasks |█▌        | 3/20 (15.0%) | ⏳ 00:28<01:54 |  6.76s/itINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNjk=/runs "HTTP/1.1 200 OK"
running tasks |██        | 4/20 (20.0%) | ⏳ 00:44<02:46 | 10.43s/itINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNjk=/runs "HTTP/1.1 200

...

running experiment evaluations |          | 0/60 (0.0%) | ⏳ 00:00<? | ?it/sINFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiment_evaluations "HTTP/1.1 200 OK"
running experiment evaluations |▏         | 1/60 (1.7%) | ⏳ 00:02<02:45 |  2.81s/itINFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:code_agent.get_routem_llm.routem_llm:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_

Experiment completed: 20 task runs, 3 evaluator runs, 60 evaluations
Rodando o agente com o modelo moonshotai/kimi-k2-instruct-0905 do provedor nvidia


c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found moonshotai/kimi-k2-instruct-0905 in available_models, but type is unknown and inference may fail.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\chat_models.py:715: UserWarning: Model 'moonshotai/kimi-k2-instruct-0905' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso
INFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/datasets/RGF0YXNldDoxMQ==/experiments "HTTP/1.1 200 OK"


🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/s/sehnemjeferson/datasets/RGF0YXNldDoxMQ==/experiments
🔗 View this experiment: https://app.phoenix.arize.com/s/sehnemjeferson/datasets/RGF0YXNldDoxMQ==/compare?experimentId=RXhwZXJpbWVudDozNzI=


running tasks |          | 0/20 (0.0%) | ⏳ 00:00<? | ?it/sINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNzI=/runs "HTTP/1.1 200 OK"
running tasks |▌         | 1/20 (5.0%) | ⏳ 00:10<03:28 | 10.95s/itINFO:httpx:HTTP Request: POST https://app.phoenix.arize.com/s/sehnemjeferson/v1/experiments/RXhwZXJpbWVudDozNzI=/runs "HTTP/1.1 200 OK"
running tasks |█         | 2/20 (10.0%) | ⏳ 00:21<03:11 | 10.64s/itINFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.cerebras.ai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:code_agent.get_routem_llm.routem_llm:Sucesso com modelo Cerebras: qwen-3-235b-a22b-instruct-2507
INFO:code_agent.get_routem_llm.routem_llm:Iniciando roteamento LLM
INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https:/